# Validation 08 — Multi-file Analysis Loop (Cell 21)
Verifies all files processed, all params computed, no NaN, sensor column correct.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, warnings
from pathlib import Path
from scipy.signal import butter,cheby1,cheby2,ellip,bessel,sosfiltfilt,sosfreqz,welch
from scipy.signal import spectrogram as sp_spectrogram
from scipy.io import wavfile
from itertools import product
warnings.filterwarnings('ignore')

BASE_DIR    = Path(r"D:\\1 placement\\IAESTE INTERNSHIP CZECH\\iaeste26-blasting-sound-main\\iaeste26-blasting-sound-main")
DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_PRIORITY   = ['AccAxial4507','AccRadial4507','Mic147EB','Mic46BE']
ENERGY_WINDOW_S   = 0.05;  NOISE_DURATION_S = 0.5;  ONSET_THRESHOLD = 10.0;  ONSET_OFFSET_S = 7.0
WINDOW_DURATION_S = 5.0;   WINDOW_STEP_S    = 1.0
LOWER_LIMITS = [176,225,283,353,440,565,707,880,1130,1414,1760,10,10,500,1000]
UPPER_LIMITS = [225,283,353,440,565,707,880,1130,1414,1760,2220,1000,2000,1500,2000]
N_BANDS      = len(LOWER_LIMITS)
BAND_LABELS  = [f"{lo}–{hi} Hz" for lo,hi in zip(LOWER_LIMITS,UPPER_LIMITS)]
FILTER_TYPES  = ['Butterworth','Chebyshev I','Chebyshev II','Elliptical','Bessel']
FILTER_ORDERS = [3,5,7]
CHEBY1_RIPPLE_DB=0.5; CHEBY2_ATTEN_DB=40.0; ELLIP_RIPPLE_DB=0.5; ELLIP_ATTEN_DB=40.0

P=[0]; F=[0]
def check(label, ok, note=""):
    s="[PASS]" if ok else "[FAIL]"
    if ok: P[0]+=1
    else:  F[0]+=1
    print(f"  {s}  {label}" + (f"  → {note}" if note else ""))
def info(label, val): print(f"  [INFO]  {label}: {val}")
def summary():
    t=P[0]+F[0]
    print(f"\n{'='*50}")
    print(f"  PASS: {P[0]}/{t}  |  FAIL: {F[0]}/{t}")
    print(f"  Score: {P[0]/t*100:.0f}%" if t else "  No checks run")
    print('='*50)
print("Config loaded.")


Config loaded.


In [2]:
def load_wav(path):
    fs,data=wavfile.read(path)
    if data.ndim>1: data=data[:,0]
    if   data.dtype==np.int16:  sig=data.astype(np.float64)/32768.0
    elif data.dtype==np.int32:  sig=data.astype(np.float64)/2147483648.0
    else:                       sig=data.astype(np.float64)
    return fs,sig,len(sig)/fs

def detect_onset(signal,fs):
    hop=int(ENERGY_WINDOW_S*fs); n=len(signal)//hop
    rms=np.array([np.sqrt(np.mean(signal[i*hop:(i+1)*hop]**2)) for i in range(n)])
    nf=np.mean(rms[:max(1,int(NOISE_DURATION_S/ENERGY_WINDOW_S))])
    ab=np.where(rms>ONSET_THRESHOLD*nf)[0]
    of=int(ab[0]) if len(ab) else int(np.argmax(rms))
    return of*hop, of*ENERGY_WINDOW_S, np.arange(n)*ENERGY_WINDOW_S, rms, nf

def make_windows(signal,fs):
    wl=int(WINDOW_DURATION_S*fs); sl=int(WINDOW_STEP_S*fs)
    n=max(0,(len(signal)-wl)//sl+1)
    return [signal[i*sl:i*sl+wl] for i in range(n)], np.arange(n)*WINDOW_STEP_S

def design_filter(ftype,order,lo,hi,fs):
    nyq=fs/2.; Wn=[lo/nyq,hi/nyq]
    if ftype=='Butterworth':  return butter(order,Wn,btype='bandpass',output='sos')
    if ftype=='Chebyshev I':  return cheby1(order,CHEBY1_RIPPLE_DB,Wn,btype='bandpass',output='sos')
    if ftype=='Chebyshev II': return cheby2(order,CHEBY2_ATTEN_DB,Wn,btype='bandpass',output='sos')
    if ftype=='Elliptical':   return ellip(order,ELLIP_RIPPLE_DB,ELLIP_ATTEN_DB,Wn,btype='bandpass',output='sos')
    if ftype=='Bessel':       return bessel(order,Wn,btype='bandpass',output='sos',norm='phase')

all_wavs = sorted(set(DATA_DIR.rglob("*.wav"))|set(DATA_DIR.rglob("*.WAV")))
def skey(p):
    s=Path(p).stem.split('_')[-1]
    return SENSOR_PRIORITY.index(s) if s in SENSOR_PRIORITY else 99
all_wavs = sorted(all_wavs,key=skey)
EXAMPLE_WAV = all_wavs[0] if all_wavs else None
print(f"WAV files found: {len(all_wavs)}")
if EXAMPLE_WAV: print(f"Using: {EXAMPLE_WAV.name}")


WAV files found: 1568
Using: G80_8_3_0_AccAxial4507.wav


In [3]:
all_wavs=sorted(set(DATA_DIR.rglob("*.wav"))|set(DATA_DIR.rglob("*.WAV")))
SELECTED=[sorted(all_wavs,key=skey)[i] for i in range(min(len(all_wavs),2))]  # small sample: O(files x 225 filters x ~64 windows)
import gc
all_results=[]; timing={}; _iter_count=0

def compute_params(x,fs):
    rms=np.sqrt(np.mean(x**2)); peak=float(np.max(np.abs(x)))
    cf=peak/rms if rms>1e-12 else 0.
    zcr=np.sum(np.abs(np.diff(np.sign(x)))>0)/(2.*(len(x)-1)/fs)
    np_=min(1024,len(x)//4); fp,psd=welch(x,fs=fs,nperseg=np_,window='hann')
    bp=float(np.trapz(psd,fp)); sc=float(np.sum(fp*psd)/max(np.sum(psd),1e-12))
    t0=time.perf_counter(); _=np.sqrt(np.mean(x**2)); ru=(time.perf_counter()-t0)*1e6
    t0=time.perf_counter(); _=float(np.max(np.abs(x)))/max(float(np.sqrt(np.mean(x**2))),1e-12); cu=(time.perf_counter()-t0)*1e6
    return {'rms':rms,'peak':peak,'crest_factor':cf,'zcr':zcr,'band_power':bp,'spectral_centroid':sc,'rms_time_us':ru,'cf_time_us':cu}

for fi,wav in enumerate(SELECTED):
    try: fs_f,sig_f,_=load_wav(wav)
    except: continue
    _,ot,_,_,_=detect_onset(sig_f,fs_f)
    us=sig_f[int((ot+ONSET_OFFSET_S)*fs_f):]
    ws,wst=make_windows(us,fs_f)
    if not ws: continue
    sensor=wav.stem.split('_')[-1]
    fc={}
    for ftype,order,(lo,hi) in product(FILTER_TYPES,FILTER_ORDERS,zip(LOWER_LIMITS,UPPER_LIMITS)):
        k=(ftype,order,lo,hi)
        try: fc[k]=design_filter(ftype,order,lo,hi,fs_f)
        except: fc[k]=None
    for ftype in FILTER_TYPES:
        for order in FILTER_ORDERS:
            for bi,(lo,hi) in enumerate(zip(LOWER_LIMITS,UPPER_LIMITS)):
                sos=fc.get((ftype,order,lo,hi))
                if sos is None: continue
                ft=[]
                for wi,w in enumerate(ws):
                    t0=time.perf_counter(); xf=sosfiltfilt(sos,w); ft.append((time.perf_counter()-t0)*1000)
                    p=compute_params(xf,fs_f)
                    _iter_count+=1
                    if _iter_count%500==0: gc.collect()
                    all_results.append({'filename':wav.name,'sensor':sensor,'file_idx':fi,
                        'filter_type':ftype,'order':order,'band_idx':bi,'band_label':BAND_LABELS[bi],
                        'low_hz':lo,'high_hz':hi,'window_idx':wi,'window_start_s':wst[wi]+ot+ONSET_OFFSET_S,
                        'filter_time_ms':ft[-1],**p})
                timing[(ftype,order,bi)]=float(np.mean(ft))

df=pd.DataFrame(all_results)
check("DataFrame non-empty",              len(df)>0,                        f"{len(df):,} rows")
check("All 6 params present",             all(c in df.columns for c in ['rms','peak','crest_factor','zcr','band_power','spectral_centroid']))
check("All 5 filter types present",       set(df['filter_type'].unique())==set(FILTER_TYPES))
check("All 3 orders present",             set(df['order'].unique())==set(FILTER_ORDERS))
check("All 15 bands present",             df['band_idx'].nunique()==N_BANDS,  f"{df['band_idx'].nunique()}/15")
check("RMS all positive",                 (df['rms']>0).all())
check("Crest Factor ≥ 1 always",         (df['crest_factor']>=1.0).all(),    f"min CF={df['crest_factor'].min():.3f}")
check("No NaN in RMS",                    df['rms'].notna().all())
check("sensor column present",            'sensor' in df.columns)
check("rms_time_us column present",       'rms_time_us' in df.columns)
check("cf_time_us column present",        'cf_time_us'  in df.columns)
check("filter_time_ms column present",    'filter_time_ms' in df.columns)
info("Files processed",  df['filename'].nunique())
info("Total rows",       f"{len(df):,}")
info("Sensors found",    list(df['sensor'].unique()))
df.to_csv(RESULTS_DIR/"blasting_all_results.csv",index=False,float_format='%.6f')
print(f"  Saved: {RESULTS_DIR/'blasting_all_results.csv'}")
summary()


  [PASS]  DataFrame non-empty  → 20,025 rows
  [PASS]  All 6 params present
  [PASS]  All 5 filter types present
  [PASS]  All 3 orders present
  [PASS]  All 15 bands present  → 15/15
  [PASS]  RMS all positive
  [PASS]  Crest Factor ≥ 1 always  → min CF=2.399
  [PASS]  No NaN in RMS
  [PASS]  sensor column present
  [PASS]  rms_time_us column present
  [PASS]  cf_time_us column present
  [PASS]  filter_time_ms column present
  [INFO]  Files processed: 2
  [INFO]  Total rows: 20,025
  [INFO]  Sensors found: ['AccAxial4507']


  Saved: D:\1 placement\IAESTE INTERNSHIP CZECH\iaeste26-blasting-sound-main\iaeste26-blasting-sound-main\results\blasting_all_results.csv

  PASS: 12/12  |  FAIL: 0/12
  Score: 100%
